
# Gegenbauer reconstruction test for a logarithmic singularity

This notebook follows `Lit_Lib/Log Singularity/logarithmic reconstruction.pdf`.
The PDF proposes testing a function on `[0,1]` with a logarithmic singularity,

$$u(x)=x\log x,$$

using the mapping

$$
y = -\frac{2\log 2}{\log(x/2)} - 1, \qquad
x(y)=2\exp\left(-\frac{2\log 2}{1+y}\right), \quad y\in[-1,1].
$$

The transformed function `u(x(y))` is smooth in `y`, so we test whether a Gegenbauer post-processing reconstruction from Fourier data is numerically feasible.


In [1]:

from pathlib import Path
import sys, time
from dataclasses import dataclass

from mpmath import mp
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Markdown

# Adjust this if the notebook is launched from another working directory.
REPO = Path.cwd()
if not (REPO / 'Gegenbauer_Python').exists():
    REPO = Path('/Users/zcq/PaperCode/Recover')
sys.path.insert(0, str(REPO))

from Gegenbauer_Python.gegenbauer_reconstruction import (
    CI,
    gegenbauer_grid_values,
    gegenbauer_norm,
    mp_format,
    rising_over_factorial,
)

mp.dps = 60
print('repo =', REPO)
print('mp.dps =', mp.dps)


repo = /Users/zcq/PaperCode/Recover
mp.dps = 60



## Core formulas from the PDF

The Fourier partial sum on `[0,1]` is

$$u_N(x)=\sum_{k=-N}^{N}\tilde u_k e^{i2\pi kx},\qquad
\tilde u_k=\int_0^1 u(x)e^{-i2\pi kx}\,dx.$$

The approximate Gegenbauer coefficients are computed by Chebyshev-Gauss quadrature for

$$
\hat g^{\lambda}(l)=\frac{1}{h_l^\lambda}\int_{-1}^{1}
(1-y^2)^{\lambda-1/2}u_N(x(y))C_l^\lambda(y)\,dy.
$$


In [2]:

def u_log(x):
    """u(x) = x log(x), with the continuous endpoint value u(0)=0."""
    x = mp.mpf(x)
    if x == 0:
        return mp.zero
    return x * mp.log(x)


def log_map_y_to_x(y, eps=mp.mpf('1e-40')):
    """x(y) = 2 exp(-2 log(2)/(1+y)); maps [-1,1] to [0,1]."""
    y = mp.mpf(y)
    if abs(1 + y) < eps:
        return mp.zero
    if abs(1 - y) < eps:
        return mp.one
    return 2 * mp.exp(-2 * mp.log(2) / (1 + y))


def log_map_x_to_y(x, eps=mp.mpf('1e-40')):
    """y(x) = -2 log(2)/log(x/2) - 1."""
    x = mp.mpf(x)
    if abs(x) < eps:
        return mp.mpf(-1)
    if abs(1 - x) < eps:
        return mp.one
    return -2 * mp.log(2) / mp.log(x / 2) - 1


# A quick consistency check for the map.
for x in [mp.mpf('0'), mp.mpf('1e-8'), mp.mpf('0.1'), mp.mpf('0.5'), mp.mpf('1')]:
    y = log_map_x_to_y(x)
    xr = log_map_y_to_x(y)
    print('x =', mp_format(x, 12), ' y =', mp_format(y, 12), ' x(y) =', mp_format(xr, 12))


x = 0.0  y = -1.0  x(y) = 0.0
x = 1.0e-8  y = -0.927471652115  x(y) = 1.0e-8
x = 0.1  y = -0.53724357368  x(y) = 0.1
x = 0.5  y = 0.0  x(y) = 0.5
x = 1.0  y = 1.0  x(y) = 1.0


In [3]:

def fourier_coefficients_unit_interval(func, N, panels=None):
    """Composite 3-point Gauss-Legendre coefficients on [0,1]."""
    panels = int(panels if panels is not None else min(160 * N, 6000))
    x1 = mp.sqrt(mp.mpf('0.6'))
    nodes = [(-x1, mp.mpf(5) / 9), (mp.zero, mp.mpf(8) / 9), (x1, mp.mpf(5) / 9)]
    h = mp.one / panels
    coeffs = {}
    for k in range(N + 1):
        acc = mp.mpc(0)
        for j in range(panels):
            a = mp.mpf(j) * h
            mid = a + h / 2
            half = h / 2
            for node, weight in nodes:
                x = mid + half * node
                acc += half * weight * func(x) * mp.exp(-CI * 2 * mp.pi * k * x)
        coeffs[k] = acc
        coeffs[-k] = mp.conj(acc)
    return coeffs


def eval_fourier_unit_interval(coeffs, x, N):
    x = mp.mpf(x)
    total = mp.mpc(coeffs[0])
    for k in range(1, N + 1):
        phase = mp.exp(CI * 2 * mp.pi * k * x)
        total += coeffs[k] * phase + coeffs[-k] / phase
    return mp.re(total)


def log_gegenbauer_coefficients_from_fourier(coeffs, N, max_m, lam, nt=None):
    """Approximate Gegenbauer coefficients g^lambda_l for l=0..max_m."""
    nt = int(nt if nt is not None else 3 * N)
    xi = [mp.zero] * (nt + 1)
    ff = [mp.zero] * (nt + 1)
    for i in range(1, nt):
        theta = mp.pi * i / nt
        xi[i] = mp.cos(theta)
        x = log_map_y_to_x(xi[i])
        ff[i] = eval_fourier_unit_interval(coeffs, x, N)

    gt = rising_over_factorial(lam, max_m)
    hg = []
    for order in range(max_m + 1):
        _, cnl = gegenbauer_grid_values(order, nt, lam, gt)
        acc = mp.zero
        for i in range(1, nt):
            # Chebyshev-Gauss: integrate (1-y^2)^(lambda-1/2) by sampling
            # C_l(y) u_N(x(y)) (1-y^2)^lambda.
            acc += cnl[i] * ff[i] * mp.power(1 - xi[i], lam) * mp.power(1 + xi[i], lam)
        hg.append(acc * mp.pi / nt / gegenbauer_norm(order, lam))
    return hg


@dataclass
class LogReconResult:
    N: int
    lam: int
    m: int
    max_error: mp.mpf
    mean_error: mp.mpf
    x_at_max: mp.mpf
    x_values: list
    approx_values: list
    exact_values: list
    errors: list


def evaluate_log_reconstruction(hg, N, lam, m, nzn=None):
    nzn = int(nzn if nzn is not None else N)
    hg = hg[: m + 1]
    gt = rising_over_factorial(lam, m)
    values = [mp.zero] * (nzn + 1)
    y_values = [mp.zero] * (nzn + 1)
    for order, coeff in enumerate(hg):
        y_grid, cnl = gegenbauer_grid_values(order, nzn, lam, gt)
        for i in range(nzn + 1):
            y_values[i] = y_grid[i]
            values[i] += coeff * cnl[i]
    x_values = [log_map_y_to_x(y) for y in y_values]
    exact = [u_log(x) for x in x_values]
    errors = [abs(a - e) for a, e in zip(values, exact)]
    imax = max(range(len(errors)), key=lambda i: errors[i])
    return LogReconResult(
        N=N,
        lam=lam,
        m=m,
        max_error=errors[imax],
        mean_error=mp.fsum(errors) / len(errors),
        x_at_max=x_values[imax],
        x_values=x_values,
        approx_values=values,
        exact_values=exact,
        errors=errors,
    )


def candidate_grid(N):
    lams = sorted({max(1, N // 32), max(1, N // 16), max(1, N // 8), max(1, N // 4)})
    ms = sorted({max(1, N // 16), max(1, N // 8), max(1, N // 4), max(1, N // 2)})
    return lams, ms



## Feasibility test

The PDF does not prescribe exact `lambda` and `m` values, so the cell below uses a small candidate grid with `lambda` and `m` proportional to `N`. The best candidate is selected for each `N` by the maximum pointwise error on the mapped Chebyshev grid.

Increase `N_VALUES` after the first smoke test. The high-precision Fourier coefficient calculation is the dominant cost.


In [6]:

N_VALUES = [20, 40, 80, 160, 320]
DPS = 60
PANELS = None  # None uses min(160*N, 6000). Increase for stricter Fourier data.

mp.dps = DPS
best_results = []
all_rows = []

for N in N_VALUES:
    started = time.time()
    lams, ms = candidate_grid(N)
    coeffs = fourier_coefficients_unit_interval(u_log, N, panels=PANELS)
    best = None
    for lam in lams:
        hg = log_gegenbauer_coefficients_from_fourier(coeffs, N, max(ms), lam)
        for m in ms:
            result = evaluate_log_reconstruction(hg, N, lam, m)
            if best is None or result.max_error < best.max_error:
                best = result
    best_results.append(best)
    all_rows.append({
        'N': N,
        'candidate_lambdas': ','.join(map(str, lams)),
        'candidate_m': ','.join(map(str, ms)),
        'best_lambda': best.lam,
        'best_m': best.m,
        'Linf_error': mp_format(best.max_error, 10),
        'L1_mean_error': mp_format(best.mean_error, 10),
        'x_at_max': mp_format(best.x_at_max, 10),
        'seconds': f'{time.time() - started:.1f}',
    })

headers = list(all_rows[0])
md = '| ' + ' | '.join(headers) + ' |\n'
md += '| ' + ' | '.join(['---'] * len(headers)) + ' |\n'
for row in all_rows:
    md += '| ' + ' | '.join(str(row[h]) for h in headers) + ' |\n'
display(Markdown(md))


| N | candidate_lambdas | candidate_m | best_lambda | best_m | Linf_error | L1_mean_error | x_at_max | seconds |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 20 | 1,2,5 | 1,2,5,10 | 1 | 10 | 0.02170919942 | 0.003427875345 | 9.997774896e-13 | 3.4 |
| 40 | 1,2,5,10 | 2,5,10,20 | 1 | 20 | 0.009682435997 | 0.001932864965 | 5.987054465e-6 | 13.1 |
| 80 | 2,5,10,20 | 5,10,20,40 | 2 | 40 | 0.01295436629 | 0.001211786552 | 0.0 | 32.1 |
| 160 | 5,10,20,40 | 10,20,40,80 | 5 | 20 | 0.02284097103 | 0.001871570886 | 0.0 | 108.3 |
| 320 | 10,20,40,80 | 20,40,80,160 | 10 | 20 | 0.1855335971 | 0.01199907681 | 0.0 | 557.1 |


In [1]:

def draw_log_error_plot(results, width=920, height=620):
    image = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    left, right, top, bottom = 85, 200, 50, 75
    x0, x1 = left, width - right
    y0, y1 = top, height - bottom
    colors = ['#c13f3f', '#2c6bb1', '#258a51', '#8a4fb2', '#d28a20']
    all_errors = [err for res in results for err in res.errors if err > 0]
    log_min = int(mp.floor(min(mp.log10(err) for err in all_errors)))
    log_max = int(mp.ceil(max(mp.log10(err) for err in all_errors)))
    if log_min == log_max:
        log_min -= 1
        log_max += 1
    floor = mp.power(10, log_min)

    def px(x):
        return int(x0 + x * (x1 - x0))

    def py(err):
        value = mp.log10(max(err, floor))
        return int(y1 - (value - log_min) * (y1 - y0) / (log_max - log_min))

    draw.rectangle([x0, y0, x1, y1], outline='black')
    draw.text((x0, 18), 'Logarithmic singularity: pointwise reconstruction error', fill='black', font=font)
    draw.text((x0, height - 35), 'x in [0,1]', fill='black', font=font)
    draw.text((10, y0 + 8), 'log10 error', fill='black', font=font)

    for tick in [0, 0.25, 0.5, 0.75, 1.0]:
        xp = px(mp.mpf(str(tick)))
        draw.line([xp, y1, xp, y1 + 5], fill='black')
        draw.text((xp - 10, y1 + 10), str(tick), fill='black', font=font)

    for exponent in range(log_min, log_max + 1):
        yp = py(mp.power(10, exponent))
        draw.line([x0 - 5, yp, x0, yp], fill='black')
        draw.line([x0, yp, x1, yp], fill='#e8e8e8')
        draw.text((x0 - 60, yp - 6), f'1e{exponent}', fill='black', font=font)

    for idx, res in enumerate(results):
        color = colors[idx % len(colors)]
        ordered = sorted(zip(res.x_values, res.errors), key=lambda item: item[0])
        pts = [(px(x), py(err)) for x, err in ordered]
        draw.line(pts, fill=color, width=2)
        lx, ly = x1 + 18, y0 + 12 + 22 * idx
        draw.line([lx, ly + 5, lx + 28, ly + 5], fill=color, width=3)
        draw.text((lx + 36, ly), f'N={res.N}, lambda={res.lam}, m={res.m}', fill='black', font=font)
    return image

img = draw_log_error_plot(best_results)
display(img)


NameError: name 'best_results' is not defined